# E1c · UCI HAR 행동 인식 실험

CNN 특징을 LSTM으로 요약해 시간 순서가 검증 성능에 도움이 되는지 확인합니다.

**실행 상태:** 드라이브 원본의 실험 코드 셀에는 저장된 실행 출력이 없습니다. 아래 결과는 실행한 것으로 간주하지 않습니다. 순서대로 실행하고 검증 지표를 기록하세요.

**데이터:** `DATA_PATH`를 현재 Colab의 `DATA/UCI-HAR Dataset` 위치에 맞춰야 합니다. 공식 test를 이용한 반복 모델 선택은 피하세요.



# 실험 E1c: CNN + LSTM 헤드 (파라미터 약 1.6만, 검증셋으로 판단)

| 셀 | 내용 | 언제 실행? |
|---|---|---|
| **셀 1** | 학습 + **검증셋** 평가 (SIT/STAND 핵심 지표 포함) | SEED 0, 1, 2 로 3번 |
| **셀 2** | **최종 테스트** (잠금: `FINAL_TEST=False`) | S 트랙·E1c 비교가 끝나고 최종 모델을 정한 뒤 **한 번만** |

> ⚠️ "모두 실행"을 누르면 셀 2는 잠금 메시지(AssertionError)를 내고 멈춥니다. 정상 동작입니다.

## 0. 라이브러리와 실험 설정

필요한 패키지를 불러오고 이 실험의 변경점과 실행 방법을 코드 주석에서 확인합니다.



In [ ]:
# ============================================================
# 실험 E1c: CNN + LSTM 헤드 (파라미터 약 1.6만) + 검증셋 기반 판단
# ------------------------------------------------------------
# 기반: CNN_UCI_HAR_S0.ipynb (E1a 모델 + 검증 평가, 테스트 잠금)
#       헤드 구조는 E1b 와 같게: [헤드] → Dropout → Dense(128) → Dropout → Dense(6)
#
# 가설 H6: CNN 이 0.2초 단위 국소 형태를 요약하고, LSTM 이 그 30개 시점을 "순서대로" 읽으면
#          평균(GAP)으로 뭉개지는 정보를 필요한 만큼 골라 담아 SIT/STAND 치우침이 줄어든다
#
# 헤드 비교 (Conv 부분은 모두 동일)
#   E1a  Flatten  시점마다 따로 연결 (위치 의존)   254,150 개
#   E1b  GAP      시점을 모두 평균 (위치 무시)      16,582 개   동적 오류↓, SIT/STAND↑(STAND 쪽 치우침)
#   E1c  LSTM(16) 시점을 순서대로 읽으며 요약        15,638 개   ← 이번 실험 (E1b 와 비슷한 크기로 맞춤)
#
# 판단 지표 (S0 과 동일): 검증 피험자 5번·17번 SIT/STAND 정확도, 보조 11·23·29번
#
# 변경 요약 (Ctrl+F 로 "[E1c 변경]" 검색)
#   1) LSTM import
#   2) 모델: Flatten() → LSTM(16)
#   3) SUMMARY·그래프·TEST 라벨 E1c
#
# 실행 방법: 셀 1 만 SEED 0 → 1 → 2 로 바꿔 3번 실행하고 SUMMARY 기록. 셀 2 는 아직 실행하지 않음
# ============================================================

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.layers import BatchNormalization, Dropout   # [E1a 변경] 규제용 층
from tensorflow.keras.layers import LSTM                         # [E1c 변경] 순서를 읽는 순환층
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping             # [E1a 변경] 조기 종료

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)




## 1. Seed

난수 시드를 고정합니다. 시드를 바꿔 재실행하면 초기화에 따른 변동을 볼 수 있습니다.



In [ ]:
# ============================================================
# 1. Seed
# ============================================================

# [E0 변경] 0, 1, 2 로 바꿔 3번 실행 (한 번의 결과는 우연일 수 있음)
SEED = 0

np.random.seed(SEED)
tf.random.set_seed(SEED)




## 2. Data path

Colab에서 Drive를 연결하고 UCI HAR 데이터의 위치를 확인합니다.



In [ ]:
# ============================================================
# 2. Data path
# ============================================================

# [E0 변경] Drive 마운트 + 프로젝트 데이터 경로
from google.colab import drive
drive.mount("/content/drive")

DATA_PATH = "/content/drive/MyDrive/가천대학교/2026-2/1. 기계학습프로그래밍/2week/DATA/UCI-HAR Dataset"

assert os.path.exists(DATA_PATH), f"데이터 경로를 확인하세요: {DATA_PATH}"




## 3. Signal names

입력으로 사용할 가속도계·자이로스코프 9개 채널의 파일명을 정의합니다.



In [ ]:
# ============================================================
# 3. Signal names
# ============================================================

signals = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]




## 4. Load dataset

공식 train/test 분할의 센서 창, 행동 라벨, 피험자 번호를 읽습니다.



In [ ]:
# ============================================================
# 4. Load dataset
# ============================================================

def load_data(split):

    X = []

    for signal in signals:

        file_path = (
            f"{DATA_PATH}/{split}/"
            f"Inertial Signals/{signal}_{split}.txt"
        )

        data = np.loadtxt(file_path)

        X.append(data)

    # (9, N, 128) -> (N, 128, 9)
    X = np.stack(X, axis=-1)

    y_path = f"{DATA_PATH}/{split}/y_{split}.txt"

    y = np.loadtxt(y_path).astype(int)

    # label 1~6 -> 0~5
    y = y - 1

    # [E0 변경] 피험자 번호(1~30) 로드
    subject_path = f"{DATA_PATH}/{split}/subject_{split}.txt"

    subject = np.loadtxt(subject_path).astype(int)

    return X, y, subject


X_train, y_train, s_train = load_data("train")
X_test, y_test, s_test = load_data("test")




## 4-1. Subject-wise validation split

피험자 5·11·17·23·29를 검증용으로 떼어 학습 피험자와 겹치지 않게 합니다.



In [ ]:
# ============================================================
# 4-1. Subject-wise validation split
# ============================================================

# [E0 변경] 피험자 5명을 통째로 검증셋으로 분리 (같은 사람이 학습·검증에 걸치지 않게)
VAL_SUBJECTS = [5, 11, 17, 23, 29]

val_mask = np.isin(s_train, VAL_SUBJECTS)

X_val, y_val = X_train[val_mask], y_train[val_mask]

# [S0 변경] 검증 윈도우의 피험자 번호 보관
# 이유: 검증 피험자별(특히 5번, 17번) 정확도를 계산하기 위함
s_val = s_train[val_mask]

X_train, y_train = X_train[~val_mask], y_train[~val_mask]


print("\nDataset shape")
print("X_train:", X_train.shape, "(16 subjects)")
print("X_val  :", X_val.shape, "subjects:", VAL_SUBJECTS)
print("X_test :", X_test.shape, "(9 subjects, 셀 2 에서만 사용)")




## 5. Normalization

학습 데이터의 평균·표준편차만 사용해 검증·테스트 입력을 변환합니다.



In [ ]:
# ============================================================
# 5. Normalization
# ============================================================

# [E0 변경] 평균/표준편차는 학습 16명으로만 계산
mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True)

X_train = (X_train - mean) / (std + 1e-8)
X_val = (X_val - mean) / (std + 1e-8)
X_test = (X_test - mean) / (std + 1e-8)




## 6. One-hot encoding

정수 행동 라벨을 6개 범주의 원핫 벡터로 바꿉니다.



In [ ]:
# ============================================================
# 6. One-hot encoding
# ============================================================

n_outputs = 6

y_train_onehot = to_categorical(
    y_train,
    num_classes=n_outputs
)

y_val_onehot = to_categorical(
    y_val,
    num_classes=n_outputs
)




## 7. CNN + LSTM Model

이 실험의 CNN 또는 CNN+LSTM 구조를 정의합니다. 비교 대상과 달라진 층은 코드 주석에 표시했습니다.



In [ ]:
# ============================================================
# 7. CNN + LSTM Model
# ============================================================

inputs = Input(shape=(128, 9))


conv1 = Conv1D(
    filters=32,
    kernel_size=3,
    activation="relu"
)(inputs)

conv1 = BatchNormalization()(conv1)   # [E1a 변경] 학습 안정화

pool1 = MaxPooling1D(
    pool_size=2
)(conv1)


conv2 = Conv1D(
    filters=64,
    kernel_size=3,
    activation="relu"
)(pool1)

conv2 = BatchNormalization()(conv2)   # [E1a 변경] 학습 안정화

pool2 = MaxPooling1D(
    pool_size=2
)(conv2)


# [E1c 변경] Flatten() → LSTM(16)
# 무엇을: Conv 두 층이 만든 (30 시점 × 64 필터) 특징을 LSTM 이 시점 순서대로 읽고,
#         마지막 상태 16개 값으로 윈도우 전체를 요약함
# 이유 1: Flatten(E1a)은 시점마다 따로 연결해 파라미터가 25만 개이고,
#         GAP(E1b)은 시점을 평균 내서 동적 오류는 줄었지만 SIT/STAND 가 STANDING 쪽으로 치우쳤음.
#         LSTM 은 순서를 반영해 요약하므로, 평균으로 뭉개지는 정보를 필요한 만큼 골라 담을 수 있음
# 이유 2: 파라미터를 E1b(16,582) 수준으로 맞춰 "헤드 방식"의 차이만 비교
#         LSTM(16) 파라미터 = 4 × (16 × (64 + 16) + 16) = 5,184 개 → 전체 약 15,638 개
#         (LSTM 의 4개 게이트가 각각 입력 64개 + 이전 상태 16개를 받아 16개를 출력)
# 예측: 동적 오류는 E1b 처럼 낮고, SIT/STAND 치우침은 E1b 보다 완화.
#       다만 정지 자세는 2.56초 안의 시간 흐름 정보가 적어, 기울기 차이가 작은 5·17번은 크게 개선되지 않을 수 있음
flat = LSTM(16)(pool2)

flat = Dropout(0.5)(flat)   # [E1a 변경] 과적합 억제


dense = Dense(
    128,
    activation="relu"
)(flat)

dense = Dropout(0.5)(dense)   # [E1a 변경] 과적합 억제


outputs = Dense(
    n_outputs,
    activation="softmax"
)(dense)




## 8. Model

모델을 만들고 손실 함수, 옵티마이저, 평가 지표를 설정합니다.



In [ ]:
# ============================================================
# 8. Model
# ============================================================

model = Model(
    inputs=inputs,
    outputs=outputs
)


model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


model.summary()




## 9. Train

학습 피험자의 데이터로 모델을 훈련하고 학습 이력을 저장합니다.



In [ ]:
# ============================================================
# 9. Train
# ============================================================

# [E1a 변경] val_loss 가 10 에폭 연속 나아지지 않으면 멈추고 최적 가중치로 복원
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train_onehot,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val_onehot),
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)




## 10. Validation evaluation   [S0 추가]

검증 피험자에 예측을 적용해 정확도·Macro F1과 혼동 행렬을 계산합니다.



In [ ]:
# ============================================================
# 10. Validation evaluation   [S0 추가]
# ============================================================

# [S0 변경] test 대신 검증셋으로 예측하고 평가
# 이유: 방법 비교·선택은 검증셋으로만 함. test 는 셀 2 에서 마지막에 한 번만
y_val_pred = np.argmax(
    model.predict(X_val, verbose=0),
    axis=1
)

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]

val_accuracy = accuracy_score(y_val, y_val_pred)

val_f1 = f1_score(y_val, y_val_pred, average="macro", zero_division=0)

cm_val = confusion_matrix(y_val, y_val_pred, labels=range(n_outputs))


print("\n==============================")
print("VALIDATION RESULTS (subjects 5, 11, 17, 23, 29)")
print("==============================")

print(f"Accuracy  : {val_accuracy:.4f}")
print(f"F1-score  : {val_f1:.4f}")

print(
    classification_report(
        y_val,
        y_val_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

print("CONFUSION MATRIX (validation)")
print(cm_val)




## 11. SIT/STAND focus on validation subjects   [S0 추가]

검증 피험자별 SIT/STAND 정확도를 보고 5·17번 핵심 지표와 나머지 지표를 분리합니다.



In [ ]:
# ============================================================
# 11. SIT/STAND focus on validation subjects   [S0 추가]
# ============================================================

# [S0 변경] 검증 피험자별 전체 정확도 + SIT/STAND 정확도
# SIT/STAND 정확도 = 실제로 앉음·섬인 윈도우 중에서 정확히 맞힌 비율 (6개 클래스 예측 기준)
# 이유: 전체 정확도에는 걷기·눕기가 섞여 있어 SIT/STAND 개선이 묻힘 → 문제 클래스만 따로 측정
SIT, STAND = 3, 4

FOCUS_SUBJECTS = [5, 17]   # 기울기 차이가 작은 검증 피험자 (5번 12.1°, 17번 14.9°)

print("\n==============================")
print("PER-SUBJECT (validation)")
print("==============================")
print("subject | all acc | SIT/STAND acc | SIT->STAND | STAND->SIT")

for s in VAL_SUBJECTS:

    m = (s_val == s)

    ms = m & np.isin(y_val, [SIT, STAND])

    sit_to_stand = np.sum(m & (y_val == SIT) & (y_val_pred == STAND))

    stand_to_sit = np.sum(m & (y_val == STAND) & (y_val_pred == SIT))

    mark = "   <- 핵심" if s in FOCUS_SUBJECTS else ""

    print(
        f"{s:7d} | {accuracy_score(y_val[m], y_val_pred[m]):.4f}  | "
        f"{accuracy_score(y_val[ms], y_val_pred[ms]):.4f}        | "
        f"{sit_to_stand:10d} | {stand_to_sit:10d}{mark}"
    )

# 핵심 지표: 5번 + 17번의 SIT/STAND 윈도우를 합쳐서 계산한 정확도
mf = np.isin(s_val, FOCUS_SUBJECTS) & np.isin(y_val, [SIT, STAND])

focus_ss_acc = accuracy_score(y_val[mf], y_val_pred[mf])

# 보조 지표: 나머지 검증 피험자(11, 23, 29)의 SIT/STAND 정확도
mo = ~np.isin(s_val, FOCUS_SUBJECTS) & np.isin(y_val, [SIT, STAND])

other_ss_acc = accuracy_score(y_val[mo], y_val_pred[mo])

val_ss_err = cm_val[SIT, STAND] + cm_val[STAND, SIT]

val_dynamic_err = cm_val[0:3, 0:3].sum() - np.trace(cm_val[0:3, 0:3])

best_epoch = int(np.argmin(history.history["val_loss"])) + 1


# [S0 변경] SUMMARY 를 검증 지표로 변경
print("\n==============================")
print("SUMMARY (결과표에 기록)")
print("==============================")

print(
    f"E1c | seed {SEED} | val acc {val_accuracy:.4f} | val F1 {val_f1:.4f} | "
    f"focus SIT/STAND(5,17) {focus_ss_acc:.4f} | other SIT/STAND(11,23,29) {other_ss_acc:.4f} | "
    f"val SIT<->STAND {val_ss_err} ({cm_val[SIT, STAND]}+{cm_val[STAND, SIT]}) | val dynamic {val_dynamic_err} | "
    f"epochs {len(history.history['val_loss'])} (best {best_epoch}) | params {model.count_params():,}"
)




## 12. Learning curve

학습 곡선을 보고 손실과 검증 지표가 에폭에 따라 어떻게 변하는지 살펴봅니다.



In [ ]:
# ============================================================
# 12. Learning curve
# ============================================================

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("epoch")
plt.title(f"E1c loss (seed {SEED})")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="train acc")
plt.plot(history.history["val_accuracy"], label="val acc")
plt.xlabel("epoch")
plt.title(f"E1c accuracy (seed {SEED})")
plt.legend()

plt.tight_layout()
plt.show()

## 🔒 최종 테스트 (한 번만 실행)
S0~S3, E1c 를 **검증 지표(5·17번 SIT/STAND 정확도)**로 비교해 최종 방법을 정한 뒤에만 실행합니다.
실행 순서: 셀 1 을 원하는 SEED 로 실행 → 아래 셀에서 `FINAL_TEST = True` 로 바꿔 실행

In [ ]:
# ============================================================
# 13. FINAL TEST   [S0 추가]
# ============================================================

# [S0 변경] 테스트 평가를 별도 셀로 분리하고 잠금
# 이유: test 점수를 보고 방법을 고르면 test 에 맞춘 선택이 되어 최종 점수의 의미가 사라짐.
#       검증셋으로 최종 방법을 정한 뒤에만 True 로 바꿔 실행 (셀 1 에서 학습한 model 을 그대로 평가)
FINAL_TEST = False

assert FINAL_TEST, (
    "테스트셋은 최종 모델을 정한 뒤 한 번만 봅니다. "
    "지금은 셀 1 의 검증 결과로 비교하세요. (최종 평가 때 FINAL_TEST = True)"
)

y_pred = np.argmax(
    model.predict(X_test, verbose=0),
    axis=1
)

accuracy = accuracy_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

cm = confusion_matrix(y_test, y_pred, labels=range(n_outputs))


print("\n==============================")
print("FINAL TEST RESULTS")
print("==============================")

print(f"Accuracy  : {accuracy:.4f}")
print(f"F1-score  : {f1:.4f}")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

print("CONFUSION MATRIX (test)")
print(cm)


print("\n==============================")
print("PER-SUBJECT ACCURACY (test)")
print("==============================")

subject_acc = {}

for s in np.unique(s_test):

    m = (s_test == s)

    subject_acc[s] = accuracy_score(y_test[m], y_pred[m])

    print(f"subject {s:2d}: {subject_acc[s]:.4f}")

worst_subject = min(subject_acc, key=subject_acc.get)

sit_stand_err = cm[SIT, STAND] + cm[STAND, SIT]

dynamic_err = cm[0:3, 0:3].sum() - np.trace(cm[0:3, 0:3])

print(
    f"\nE1c TEST | seed {SEED} | acc {accuracy:.4f} | macro F1 {f1:.4f} | "
    f"SIT<->STAND {sit_stand_err} | dynamic {dynamic_err} | "
    f"worst subject {worst_subject} ({subject_acc[worst_subject]:.4f})"
)